# Suite BND — Typed bindings

`Bindings` binds a program's own class to a reference object schema: the class gives `read(instance)` and
`make(state)`, between its fields and the schema's properties and entries, and the binding does the rest: `accept`,
a builder that implements `Visitors.OfObject` over a state, and a registry for `Plain.FromPlain`. A binding may fix a
property's value, make properties exclusive, and leave an adjacency implied by its other ends.

In [ ]:
import { Bindings as Bd, Comparison as C, JSON, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { AttributeError, KeyError, LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises, same, text } from "./support.js";

const store = new Proxies.OfStore();

const Note = new S.OfObject.Builder().properties(text("text")).create();
const Link = new S.OfRelation.Builder().links("source", "target").properties(text("weight", BigInt),
  (p) => p.name("note").of(Note)).create();
const Shape = new S.OfObject.Builder().properties(text("w", BigInt)).create();
const Node = new S.OfObject.Builder().ref().properties(
  text("tag"), text("label"), text("i", BigInt), text("f", Number), (p) => p.name("shape").of(Shape),
  (p) => p.name("tags").of((t) => t.as_indexed((i) => i.of((t) => t.as_native(String))))).relations(
  (a) => a.name("out").of(Link).me("source"), (a) => a.name("into").of(Link).me("target")).create();
store.register("Node", Node);
store.register("Link", Link);

type Out = [Point, bigint | null, any];

/** A class of the program's own, bound to `Node`: its number is `i` or `f`, its shape and tags are plain data, and
 * `out` holds (target, weight, note) triples. */
class Point {
  constructor(public label: string | null = null, public number: bigint | number | null = null, public shape: any = null,
    public tags: any = null, public out: Out[] = []) {}

  identity(): unknown {
    return this;
  }

  schema_name(): string {
    return "Node";
  }

  owner(): null {
    return null;
  }

  accept(visitor: any): void {
    Bd.accept(BINDING, this, visitor);
  }
}

function read(point: Point): Bd.State {
  const values = new Map<string, unknown>([["label", point.label], ["shape", point.shape], ["tags", point.tags]]);
  if (point.number !== null) values.set(typeof point.number === "bigint" ? "i" : "f", point.number);
  return new Bd.State(values, new Map([["out", point.out.map(([t, w, n]) =>
    new Bd.Entry(new Map([["target", t]]), new Map<string, unknown>([["weight", w], ["note", n]])))]]));
}

function make(state: Bd.State): Point {
  const v = state.values;
  const get = (name: string): any => v.get(name) ?? null;
  return new Point(get("label"), get("i") ?? get("f"), get("shape"), get("tags"),
    (state.entries.get("out") ?? []).map((e) => [e.links.get("target") ?? null, e.properties.get("weight") ?? null,
      e.properties.get("note") ?? null] as Out));
}

function assign(point: Point, state: Bd.State): Point {
  return Object.assign(point, make(state));
}

const BINDING = new Bd.Binding(Node, read, make, assign,
  { fixed: new Map([["tag", "node"]]), exclusive: [["i", "f"]], implied: ["into"] });
const B = new Bd.OfStore(new Map([["Node", [Node, (instance?: Point) => new Bd.Builder(BINDING, instance)] as const]]),
  new Map([["Link", Link]]));
const native = (value: unknown) => (p: any) => p.value((a: any) => a.as_native((n: any) => n.set(value)));

## BND-01 · A bound class writes itself through its binding, and is rebuilt from snapshots with the registry

In [ ]:
{
  const b = new Point("b", 1.5, null, ["x"]);
  const a = new Point("a", 5n, map({ w: 3n }), ["p", "q"], [[b, 7n, map({ text: "hi" })], [b, null, null]]);
  const text_ = JSON.ToJSON(store).Reachable(Node, a);
  assert(text_ ===
    '{"root": "s0", "objects": {"s0": {"tag": "node", "label": "a", "i": 5, "shape": {"w": 3}, "tags": ["p", "q"], ' +
    '"out": [{"target": {"$ref": "s1", "$schema": "Node"}, "weight": 7, "note": {"text": "hi"}}, ' +
    '{"target": {"$ref": "s1", "$schema": "Node"}}]}, "s1": {"tag": "node", "label": "b", "f": 1.5, "tags": ["x"]}}}');
  const back = JSON.FromJSON(B).Reachable(Node, text_) as Point;
  assert(back instanceof Point && back.number === 5n && equal(back.shape, map({ w: 3n })) && same(back.tags, ["p", "q"]));
  assert(back.out[0]![0].number === 1.5 && back.out[0]![1] === 7n && equal(back.out[0]![2], map({ text: "hi" })));
  assert(back.out[1]![1] === null && back.out[1]![2] === null);
  assert(JSON.ToJSON(store).Reachable(Node, back) === text_);
  assert(YAML.ToYAML(store).Reachable(Node, YAML.FromYAML(B).Reachable(Node, YAML.ToYAML(store).Reachable(Node, a)) as Point) ===
    YAML.ToYAML(store).Reachable(Node, a));
  assert(Validators.Validate(B).Reachable(Node, back).length === 0);
  assert(new C.OfObject(Node, a).compare(new C.OfObject(Node, back)) === 0);
}

## BND-02 · A builder holds a state: natives checked by type, fixed and exclusive properties, other values as plain data

In [ ]:
{
  const a = new Point("a", 5n, map({ w: 3n }), ["p", "q"]);
  const builder = new Bd.Builder(BINDING, a);
  const names: string[] = [];
  builder.properties((p) => names.push(p.name()));
  assert(same(names, ["tag", "label", "i", "shape", "tags"]));
  assert(builder.has("tag") && builder.has("i") && !builder.has("f") && !builder.has("nope"));
  assert(!new Bd.Builder(BINDING, new Point()).state.values.has("label")); // a builder drops absent values
  // Natives: set, read back, type-checked, cleared.
  builder.property("label", native("a2"));
  const got: unknown[] = [];
  builder.property("label", (p) => p.value((v) => v.as_native((n) => got.push([n.has(), n.get()]))));
  assert(equal(got, [[true, "a2"]]));
  raises(TypeError, () => builder.property("label", native(1n)), "expected str, got int");
  raises(AttributeError, () => builder.property("f", (p) => p.value((v) => v.as_native((n) => n.get()))),
    "property 'f' is not set");
  for (const method of ["as_object", "as_union", "as_intersection", "as_indexed"]) {
    raises(TypeError, () => builder.property("label", (p) => p.value((v) => (v as any)[method](() => null))),
      "property 'label' is native");
  }
  // Exclusive: writing one clears the others of its group.
  builder.property("f", native(2.5));
  assert(!builder.has("i") && builder.clone().number === 2.5);
  // Fixed: it always holds the binding's value, writing another throws, and clearing changes nothing.
  builder.property("tag", native("node"));
  raises(ValueError, () => builder.property("tag", native("other")), "expected tag 'node', got 'other'");
  raises(TypeError, () => builder.property("tag", native(1n)), "expected str, got int");
  builder.clear("tag").property("tag", (p) => p.clear());
  assert(builder.has("tag"));
  // Other values, through their plain form: a value object, a list.
  builder.property("shape", (p) => p.value((v) => v.as_object((o) => o.property("w", native(9n)))));
  builder.property("tags", (p) => p.value((v) => v.as_indexed((l) => l.append((i) => i.as_native((n) => n.set("r"))))));
  let made = builder.clone();
  assert(equal(made.shape, map({ w: 9n })) && same(made.tags, ["p", "q", "r"]) && made.label === "a2");
  builder.property("shape", (p) => p.clear()).clear("tags").clear("nope").property("label", (p) => p.clear());
  made = builder.clone();
  assert(made.shape === null && made.tags === null && made.label === null);
  raises(KeyError, () => builder.property("nope", () => null), "unknown property 'nope'");
}

## BND-03 · A builder reads and adds entries; an implied adjacency keeps none

In [ ]:
{
  const b = new Point("b");
  const c = new Point("c");
  const builder = new Bd.Builder(BINDING, new Point("a", null, null, null, [[b, 7n, map({ text: "hi" })]]));
  const names: unknown[] = [];
  builder.adjacencies((adj) => names.push([adj.name(), adj.me()]));
  assert(equal(names, [["out", "source"], ["into", "target"]]));
  const seen: unknown[] = [];
  builder.adjacency("out", (adj) => adj.entries((e) => e.links((k) => { seen.push(k.name()); k.target((t) => seen.push(t)); })
    .properties((p) => seen.push(p.name()))));
  assert(same(seen, ["target", b, "weight", "note"]));
  const fill = (e: any) => {
    e.link("target", (k: any) => k.set(c)).property("weight", native(1n));
    assert(e.has("weight") && !e.has("note") && !e.has("nope"));
    e.clear("note");
  };
  builder.adjacency("out", (adj) => adj.add(fill));
  assert(equal(builder.clone().out.map(([t, w]: Out) => [t.label, w]), [["b", 7n], ["c", 1n]]));
  const props: string[] = [];
  builder.adjacency("out", (adj) => adj.entries((e) => e.properties((p) => props.push(p.name()))));
  assert(same(props, ["weight", "note", "weight"])); // an absent property is not visited
  const wrong: [(e: any) => unknown, Function, string][] = [
    [(e) => e.link("source", () => null), KeyError, "'source' is not a link this entry can set"],
    [(e) => e.property("nope", () => null), KeyError, "unknown property 'nope'"],
    [(e) => e.link("target", (k: any) => k.target((t: unknown) => seen.push(t))), ValueError, "link 'target' is not set"],
  ];
  for (const [w, error, message] of wrong) raises(error, () => builder.adjacency("out", (adj) => adj.add(w)), message);
  builder.adjacency("out", (adj) => adj.entries((e) => e.link("target", (k) => k.target(
    (t) => { if (t === b) adj.remove(e); }))).remove(null as any)); // removing what is not an entry changes nothing
  assert(same(builder.clone().out.map(([t]: Out) => t.label), ["c"]));
  // An implied adjacency: the other ends imply its entries, so those added are ignored, and none are read.
  builder.adjacency("into", (adj) => adj.add((e) => e.link("source", (k) => k.set(c))).entries((e) => seen.push(e)).remove(null as any));
  assert(!builder.state.entries.has("into"));
  raises(KeyError, () => builder.adjacency("nope", () => null), "unknown adjacency 'nope'");
  assert(builder.identify(b) === builder);
}

## BND-04 · Finalizing a builder, and the registry

In [ ]:
{
  const a = new Point("a");
  assert(new Bd.Builder(BINDING).create() instanceof Point && new Bd.Builder(BINDING, a).clone() !== a);
  const updated = new Bd.Builder(BINDING, a);
  updated.property("label", native("a2"));
  assert(updated.update() === a && a.label === "a2");
  raises(ValueError, () => new Bd.Builder(BINDING, a).create(),
    "create() is only valid without a source instance; use clone() or update()");
  raises(ValueError, () => new Bd.Builder(BINDING).clone(), "clone() is only valid with a source instance");
  raises(ValueError, () => new Bd.Builder(BINDING).update(), "update() is only valid with a source instance");
  assert(equal([...new Bd.Builder(BINDING).adjacencies(() => null).state.entries], [["out", []]])); // an implied one has none
  assert(new Bd.Binding(Node, read, make).assign(a, new Bd.State()) === a); // by default, an instance is not written back
  // The registry: builders by schema name, as decoding needs them.
  assert(B.schema("Node") === Node && B.name_of(Node) === "Node" && B.name_of(Link) === "Link" && B.member(a, "label") === "a2");
  assert((B["Node"] as any)() instanceof Bd.Builder && (B["Node"] as any)(a) instanceof Bd.Builder);
  raises(TypeError, () => B.schema("Link"), "'Link' is a relation; no relation builder is exposed");
  raises(AttributeError, () => B.schema("nope"), "no schema registered as 'nope'");
  raises(LookupError, () => B.name_of(Shape), "schema is not registered");
  for (const name of ["nope", "_private"]) raises(AttributeError, () => B[name]);
  // A fixed value is compared as a value, bytes too; a registry may know no relations.
  const Blob = new S.OfObject.Builder().ref().properties(text("b", Uint8Array)).create();
  const blobs = new Bd.OfStore(new Map([["Blob", [Blob, () => new Bd.Builder(
    new Bd.Binding(Blob, read, make, null, { fixed: new Map([["b", new Uint8Array([1])]]) }))] as const]]));
  (blobs["Blob"] as any)().property("b", native(new Uint8Array([1])));
  raises(ValueError, () => (blobs["Blob"] as any)().property("b", native(new Uint8Array([2]))),
    "expected b b'\\x01', got b'\\x02'");
  raises(ValueError, () => (blobs["Blob"] as any)().property("b", native(new Uint8Array([1, 2]))), "got b'\\x01\\x02'");
  assert(blobs.name_of(Blob) === "Blob");
}